In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np


In [2]:
# SETUP

TRAINING_DATA = "~/Downloads/training_data_clean.csv"


# def prep_data():
df = pd.read_csv(TRAINING_DATA)

# rename
df.columns = [
    'id',
    'best_tasks_free',
    'acad_tasks_rating',
    'best_tasks_select',
    'subopt_freq_rating',
    'subopt_tasks_select',
    'subopt_tasks_free',
    'evidence_freq_rating',
    'verify_freq_rating',
    'verify_method_free',
    'target'
    ]

for task in df['best_tasks_select'].unique():
    print(task)

# prep_data()


Math computations,Data processing or analysis,Explaining complex concepts simply,Writing or editing essays/reports,Drafting professional text (e.g., emails, résumés)
Writing or debugging code
Math computations,Converting content between formats (e.g., LaTeX)
Explaining complex concepts simply,Converting content between formats (e.g., LaTeX),Writing or editing essays/reports,Drafting professional text (e.g., emails, résumés),Brainstorming or generating creative ideas
Math computations,Writing or debugging code,Data processing or analysis,Converting content between formats (e.g., LaTeX)
Brainstorming or generating creative ideas
Writing or debugging code,Data processing or analysis,Explaining complex concepts simply,Converting content between formats (e.g., LaTeX),Writing or editing essays/reports,Drafting professional text (e.g., emails, résumés),Brainstorming or generating creative ideas
nan
Explaining complex concepts simply,Converting content between formats (e.g., LaTeX),Brainstormi

In [3]:
# def split_data(df):
students = df['id'].unique()
train_idx, test_idx = train_test_split(
    students, test_size=0.3, random_state=22
)

train_df = df[df['id'].isin(train_idx)]
test_df = df[df['id'].isin(test_idx)]

train_groups = train_df['id'].values
test_groups = test_df['id'].values

x_train = train_df.drop(columns=['target', 'id'])
y_train = train_df['target']

x_test = test_df.drop(columns=['target', 'id'])
y_test = test_df['target']

    # return x_train, x_test, y_train, y_test, train_groups, test_groups

In [4]:
# EXPLORE DATA

df.isnull().sum()

id                       0
best_tasks_free          2
acad_tasks_rating        0
best_tasks_select       15
subopt_freq_rating       1
subopt_tasks_select     22
subopt_tasks_free       11
evidence_freq_rating    62
verify_freq_rating       4
verify_method_free      10
target                   0
dtype: int64

In [ ]:
# PREPROCESSING 
# """These are EQUIVALENT:

# Option 1: make_pipeline (shorter)
# text_pipeline = make_pipeline(
#     SimpleImputer(strategy="constant", fill_value=""),
#     TfidfVectorizer(max_features=2000)
# )

# Option 2: Pipeline (explicit)
# text_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy="constant", fill_value="")),
#     ('tfidf', TfidfVectorizer(max_features=2000))
# ])"""

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer, OrdinalEncoder
from typing import Dict, List
import unicodedata


text_cols = [
    'best_tasks_free',
    'subopt_tasks_free',
    'verify_method_free',
    ]


ord_cols = [
    'acad_tasks_rating',
    'subopt_freq_rating',
    'evidence_freq_rating',
    'verify_freq_rating',
    ]

likert_categories = [
    '1 — Not at all likely',
    '2 — Unlikely',
    '3 — Neutral / Unsure',
    '4 — Likely',
    '5 — Very likely'
]

freq_categories = [
    '1 — Never',
    '2 — Rarely',
    '3 — Sometimes',
    '4 — Often',
    '5 — Very often'
]

ord_categories = [
    likert_categories,  # acad_tasks_rating
    freq_categories,    # subopt_freq_rating
    freq_categories,    # evidence_freq_rating
    freq_categories,    # verify_freq_rating
]

cat_cols = [
    'best_tasks_select',
    'subopt_tasks_select',
    ]

cat_multi_select_choices = [
    "Explaining complex concepts simply",
    "Drafting professional text (e.g., emails, résumés)",
    "Brainstorming or generating creative ideas",
    "Writing or editing essays/reports",
    "Converting content between formats (e.g., LaTeX)",
    "Writing or debugging code",
    "Data processing or analysis",
    "Math computations",
]

# custom function makecorpus just to keep consistency in pipeline
class MakeCorpus(BaseEstimator, TransformerMixin):
    # required by TansformerMixin -ignore
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        # X is DataFrame with text columns
        return X.agg(' '.join, axis=1).tolist()

def preprocess_text():
    # TFIDF manually (classical)
    return Pipeline([
        ('imputer', SimpleImputer(strategy="constant", fill_value="")),
        ('combiner', MakeCorpus()),
        # hardcoded stop_words param for now
        # TODO: try with max instead
        ('tfidf', TfidfVectorizer(stop_words='english'))
    ])

def preprocess_ord():

    return make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OrdinalEncoder(categories=ord_categories)

    )

def _normalize(s: str) -> str:
    # normalize unicode, collapse spaces, strip
    s = unicodedata.normalize("NFKC", str(s))
    s = " ".join(s.split())
    return s.strip()

def _split_top_level_commas(s: str) -> List[str]:
    """
    Split on commas that are NOT inside parentheses.
    Example: "Drafting ... (e.g., emails, résumés), Math computations"
    -> ["Drafting ... (e.g., emails, résumés)", "Math computations"]
    """
    parts = []
    buf = []
    depth = 0
    for ch in s:
        if ch == '(':
            depth += 1
            buf.append(ch)
        elif ch == ')':
            depth = max(0, depth - 1)
            buf.append(ch)
        elif ch == ',' and depth == 0:
            parts.append("".join(buf))
            buf = []
        else:
            buf.append(ch)
    if buf:
        parts.append("".join(buf))
    return parts

class MultiSelectBinarizerPerColumn(BaseEstimator, TransformerMixin):
    """
    One-hot encode multi-select columns using a safe split and a fixed
    set of canonical choices. Clone-safe: __init__ does not modify params.
    If the original cell is NaN -> all dummies for that column are NaN.
    """
    def __init__(self, classes: List[str]):
        self.classes = classes  # DO NOT MODIFY HERE (clone-safe)

    # internal helpers use fitted attributes
    def _parse_cell(self, x) -> List[str]:
        if pd.isna(x) or (isinstance(x, str) and x.strip() == ""):
            return []
        norm = _normalize(x)
        pieces = [_normalize(p) for p in _split_top_level_commas(norm)]
        # keep only exact matches (normalized)
        return [p for p in pieces if p in self.classes_]

    def fit(self, X, y=None):
        X = pd.DataFrame(X)

        # create normalized, fitted classes (separate from init param)
        self.classes_ = [_normalize(c) for c in self.classes]

        self.mlbs_: Dict[str, MultiLabelBinarizer] = {}
        self.col_to_outcols_: Dict[str, List[str]] = {}

        for col in X.columns:
            mlb = MultiLabelBinarizer(classes=self.classes_)
            mlb.fit([[]])  # fit on fixed classes only
            self.mlbs_[col] = mlb
            self.col_to_outcols_[col] = [f"{col}__{c}" for c in mlb.classes_]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        blocks = []
        for col in X.columns:
            mlb = self.mlbs_[col]
            outcols = self.col_to_outcols_[col]
            is_missing = X[col].isna()
            lists = X[col].apply(self._parse_cell)
            arr = mlb.transform(lists)
            df_bin = pd.DataFrame(arr, columns=outcols, index=X.index)
            df_bin.loc[is_missing, :] = np.nan  # preserve missingness for KNN later
            blocks.append(df_bin)
        return pd.concat(blocks, axis=1)

def preprocess_cat():
    return make_pipeline(MultiSelectBinarizerPerColumn(classes=cat_multi_select_choices),
                                 SimpleImputer(strategy="constant", fill_value=0))

def create_preprocessor():
    # create pipelines for each type, applied per column within each subset
    # pipelines run in tandem
    # changed names to shorter versions for param_grid referencing
    transformers = [
        ("text", preprocess_text(), text_cols), # pass in just the names of the columns for now, df specified later in full pipeline
        ("ord", preprocess_ord(), ord_cols),
         ("cat", preprocess_cat(), cat_cols),
    ]

    return ColumnTransformer(transformers=transformers)


In [ ]:
def run_pipeline()
preprocessor = create_preprocessor()   

# added logistic regression as placeholder classifier for now
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

param_grid = {
    # Text TF-IDF parameters

    'preprocessor__text__tfidf__max_features': [100, 200, 500, 1000, 2000],
    'preprocessor__text__tfidf__ngram_range': [(1, 1), (1, 2)],
    'preprocessor__text__tfidf__min_df': [2, 3, 4],
    'preprocessor__text__tfidf__max_df': [0.75, 0.9]


    # Classifier parameters
    # TODO: change to SVM or RandomForest and tune their hyperparams
}

# Grid search
grid_search = GridSearchCV(full_pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

# Fit on raw data (pipeline handles preprocessing)

full_pipeline.fit(x_train, y_train)

# y_pred = full_pipeline.predict(x_train)
# print(f"Train accuracy: {np.mean(y_pred == y_train)}")
# df specified here
grid_search.fit(x_train, y_train)

y_pred_train = grid_search.predict(x_train)
train_acc = np.mean(y_pred_train == y_train)
print(f"Grid Search - Training accuracy: {train_acc:.3f}")
print(f"Grid Search - Best CV score: {grid_search.best_score_:.3f}")
print(f"Best parameters:{grid_search.best_params_}")

In [18]:
# ALL MODELS COMBINED, GRIDSEARCH

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
import numpy as np
from xgboost import XGBClassifier

# ============================================
# SETUP: GroupKFold cross-validation
# ============================================
cv_groups = GroupKFold(n_splits=5)

# Get the best preprocessor from your earlier grid search
# (assuming you've already run the initial grid_search)
best_preprocessor = grid_search.best_estimator_.named_steps['preprocessor']

print("="*60)
print("TESTING MODELS WITH GROUPKFOLD")
print("="*60)

# ============================================
# MODEL 1: LOGISTIC REGRESSION
# ============================================
print("\n### 1. LOGISTIC REGRESSION ###")

pipeline_logreg = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

param_grid_logreg = {
    'classifier__C': [0.001, 0.01, 0.1, 1.0, 10.0]
}

grid_logreg = GridSearchCV(
    pipeline_logreg,
    param_grid_logreg,
    cv=cv_groups,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_logreg.fit(x_train, y_train, groups=train_groups)

print(f"Best params: {grid_logreg.best_params_}")
print(f"Best VALIDATION score: {grid_logreg.best_score_:.3f}")
print(f"Training score: {grid_logreg.score(x_train, y_train):.3f}")
print(f"Overfit gap: {grid_logreg.score(x_train, y_train) - grid_logreg.best_score_:.3f}")

# ============================================
# MODEL 2: LINEAR SVC
# ============================================
print("\n### 2. LINEAR SVC ###")

pipeline_svc = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', LinearSVC(dual=True, max_iter=10000, class_weight='balanced'))
])

param_grid_svc = {
    'classifier__C': [0.01, 0.1, 1.0, 10.0]
}

grid_svc = GridSearchCV(
    pipeline_svc,
    param_grid_svc,
    cv=cv_groups,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_svc.fit(x_train, y_train, groups=train_groups)

print(f"Best params: {grid_svc.best_params_}")
print(f"Best VALIDATION score: {grid_svc.best_score_:.3f}")
print(f"Training score: {grid_svc.score(x_train, y_train):.3f}")
print(f"Overfit gap: {grid_svc.score(x_train, y_train) - grid_svc.best_score_:.3f}")

# ============================================
# MODEL 3: NAIVE BAYES
# ============================================
print("\n### 3. NAIVE BAYES ###")

pipeline_nb = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', MultinomialNB())
])

param_grid_nb = {
    'classifier__alpha': [0.1, 0.5, 1.0, 2.0, 5.0]
}

grid_nb = GridSearchCV(
    pipeline_nb,
    param_grid_nb,
    cv=cv_groups,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_nb.fit(x_train, y_train, groups=train_groups)

print(f"Best params: {grid_nb.best_params_}")
print(f"Best VALIDATION score: {grid_nb.best_score_:.3f}")
print(f"Training score: {grid_nb.score(x_train, y_train):.3f}")
print(f"Overfit gap: {grid_nb.score(x_train, y_train) - grid_nb.best_score_:.3f}")

# ============================================
# MODEL 4: RANDOM FOREST
# ============================================
print("\n### 4. RANDOM FOREST ###")

pipeline_rf = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=22))
])

param_grid_rf = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [5, 10, 20],
    'classifier__min_samples_leaf': [5, 10, 20],
    'classifier__max_features': ['sqrt', 'log2']
}

grid_rf = GridSearchCV(
    pipeline_rf,
    param_grid_rf,
    cv=cv_groups,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(x_train, y_train, groups=train_groups)

print(f"Best params: {grid_rf.best_params_}")
print(f"Best VALIDATION score: {grid_rf.best_score_:.3f}")
print(f"Training score: {grid_rf.score(x_train, y_train):.3f}")
print(f"Overfit gap: {grid_rf.score(x_train, y_train) - grid_rf.best_score_:.3f}")

# ============================================
# MODEL 5: XGBOOST
# ============================================
from sklearn.preprocessing import LabelEncoder

# Encode the labels BEFORE splitting
le = LabelEncoder()
df['target_encoded'] = le.fit_transform(df['target'])

# Now split with encoded target
def split_data(df):
    students = df['id'].unique()
    train_idx, test_idx = train_test_split(
        students, test_size=0.3, random_state=22
    )

    train_df = df[df['id'].isin(train_idx)]
    test_df = df[df['id'].isin(test_idx)]

    train_groups = train_df['id'].values
    test_groups = test_df['id'].values

    x_train = train_df.drop(columns=['target', 'target_encoded', 'id'])
    y_train = train_df['target_encoded']  # Use encoded version

    x_test = test_df.drop(columns=['target', 'target_encoded', 'id'])
    y_test = test_df['target_encoded']  # Use encoded version

    return x_train, x_test, y_train, y_test, train_groups, test_groups

# Split the data
x_train, x_test, y_train, y_test, train_groups, test_groups = split_data(df)

print("\n### 5. XGBOOST ###")
pipeline_xgb = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', XGBClassifier(
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=22
    ))
])

param_grid_xgb = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3,5, 7],
    'classifier__learning_rate': [0.01,  0.1, 0.3],
    'classifier__subsample': [0.8, 1.0],
    'classifier__colsample_bytree': [0.8, 1.0],
    'classifier__min_child_weight': [1, 3, 5]
}

grid_xgb = GridSearchCV(
    pipeline_xgb,
    param_grid_xgb,
    cv=cv_groups,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)


grid_xgb.fit(x_train, y_train, groups=train_groups)

print(f"Best params: {grid_xgb.best_params_}")
print(f"Best VALIDATION score: {grid_xgb.best_score_:.3f}")
print(f"Training score: {grid_xgb.score(x_train, y_train):.3f}")
print(f"Overfit gap: {grid_xgb.score(x_train, y_train) - grid_xgb.best_score_:.3f}")




TESTING MODELS WITH GROUPKFOLD

### 1. LOGISTIC REGRESSION ###
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best params: {'classifier__C': 0.1}
Best VALIDATION score: 0.667
Training score: 0.707
Overfit gap: 0.040

### 2. LINEAR SVC ###
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best params: {'classifier__C': 0.1}
Best VALIDATION score: 0.674
Training score: 0.804
Overfit gap: 0.130

### 3. NAIVE BAYES ###
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best params: {'classifier__alpha': 0.5}
Best VALIDATION score: 0.656
Training score: 0.757
Overfit gap: 0.100

### 4. RANDOM FOREST ###
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best params: {'classifier__max_depth': 5, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 5, 'classifier__n_estimators': 100}
Best VALIDATION score: 0.677
Training score: 0.773
Overfit gap: 0.095

### 5. XGBOOST ###
Fitting 5 folds for each of 324 candidates, totalling 1620 fits


c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\xgboost\training.py:199: UserWarning: [11:20:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best params: {'classifier__colsample_bytree': 1.0, 'classifier__learning_rate': 0.1, 'classifier__max_depth': 5, 'classifier__min_child_weight': 1, 'classifier__n_estimators': 100, 'classifier__subsample': 0.8}
Best VALIDATION score: 0.675
Training score: 0.990
Overfit gap: 0.314


In [19]:
# ALL MODELS COMBINED, OPTUNA OPTIMIZATION

from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
import numpy as np
import optuna
from optuna.samplers import TPESampler

# ============================================
# SETUP: GroupKFold cross-validation
# ============================================
cv_groups = GroupKFold(n_splits=5)

# Get the best preprocessor from your earlier grid search
# (assuming you've already run the initial grid_search)
best_preprocessor = grid_search.best_estimator_.named_steps['preprocessor']

print("="*60)
print("TESTING MODELS WITH OPTUNA OPTIMIZATION")
print("="*60)

# ============================================
# MODEL 1: LOGISTIC REGRESSION
# ============================================
print("\n### 1. LOGISTIC REGRESSION ###")

def objective_logreg(trial):
    """Objective function for Logistic Regression"""
    C = trial.suggest_float('C', 0.001, 10.0, log=True)
    
    pipeline = Pipeline([
        ('preprocessor', best_preprocessor),
        ('classifier', LogisticRegression(
            C=C,
            max_iter=1000,
            class_weight='balanced',
            random_state=22
        ))
    ])
    
    scores = cross_val_score(
        pipeline, x_train, y_train,
        cv=cv_groups.split(x_train, y_train, groups=train_groups),
        scoring='accuracy',
        n_jobs=-1
    )
    
    return scores.mean()

study_logreg = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=22)
)
study_logreg.optimize(objective_logreg, n_trials=50, show_progress_bar=True)

# Train final model with best params
best_logreg = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', LogisticRegression(
        C=study_logreg.best_params['C'],
        max_iter=1000,
        class_weight='balanced',
        random_state=22
    ))
])
best_logreg.fit(x_train, y_train)

print(f"Best params: {study_logreg.best_params}")
print(f"Best VALIDATION score: {study_logreg.best_value:.3f}")
print(f"Training score: {best_logreg.score(x_train, y_train):.3f}")
print(f"Overfit gap: {best_logreg.score(x_train, y_train) - study_logreg.best_value:.3f}")

# ============================================
# MODEL 2: LINEAR SVC
# ============================================
print("\n### 2. LINEAR SVC ###")

def objective_svc(trial):
    """Objective function for Linear SVC"""
    C = trial.suggest_float('C', 0.01, 10.0, log=True)
    
    pipeline = Pipeline([
        ('preprocessor', best_preprocessor),
        ('classifier', LinearSVC(
            C=C,
            dual=True,
            max_iter=10000,
            class_weight='balanced',
            random_state=22
        ))
    ])
    
    scores = cross_val_score(
        pipeline, x_train, y_train,
        cv=cv_groups.split(x_train, y_train, groups=train_groups),
        scoring='accuracy',
        n_jobs=-1
    )
    
    return scores.mean()

study_svc = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=22)
)
study_svc.optimize(objective_svc, n_trials=50, show_progress_bar=True)

# Train final model with best params
best_svc = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', LinearSVC(
        C=study_svc.best_params['C'],
        dual=True,
        max_iter=10000,
        class_weight='balanced',
        random_state=22
    ))
])
best_svc.fit(x_train, y_train)

print(f"Best params: {study_svc.best_params}")
print(f"Best VALIDATION score: {study_svc.best_value:.3f}")
print(f"Training score: {best_svc.score(x_train, y_train):.3f}")
print(f"Overfit gap: {best_svc.score(x_train, y_train) - study_svc.best_value:.3f}")

# ============================================
# MODEL 3: NAIVE BAYES
# ============================================
print("\n### 3. NAIVE BAYES ###")

def objective_nb(trial):
    """Objective function for Naive Bayes"""
    alpha = trial.suggest_float('alpha', 0.1, 5.0)
    
    pipeline = Pipeline([
        ('preprocessor', best_preprocessor),
        ('classifier', MultinomialNB(alpha=alpha))
    ])
    
    scores = cross_val_score(
        pipeline, x_train, y_train,
        cv=cv_groups.split(x_train, y_train, groups=train_groups),
        scoring='accuracy',
        n_jobs=-1
    )
    
    return scores.mean()

study_nb = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=22)
)
study_nb.optimize(objective_nb, n_trials=50, show_progress_bar=True)

# Train final model with best params
best_nb = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', MultinomialNB(alpha=study_nb.best_params['alpha']))
])
best_nb.fit(x_train, y_train)

print(f"Best params: {study_nb.best_params}")
print(f"Best VALIDATION score: {study_nb.best_value:.3f}")
print(f"Training score: {best_nb.score(x_train, y_train):.3f}")
print(f"Overfit gap: {best_nb.score(x_train, y_train) - study_nb.best_value:.3f}")

# ============================================
# MODEL 4: RANDOM FOREST
# ============================================
print("\n### 4. RANDOM FOREST ###")

def objective_rf(trial):
    """Objective function for Random Forest"""
    n_estimators = trial.suggest_int('n_estimators', 100, 300)
    max_depth = trial.suggest_int('max_depth', 5, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 20)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])
    
    pipeline = Pipeline([
        ('preprocessor', best_preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            class_weight='balanced',
            random_state=22,
            n_jobs=1  # Set to 1 since we're using n_jobs=-1 in cross_val_score
        ))
    ])
    
    scores = cross_val_score(
        pipeline, x_train, y_train,
        cv=cv_groups.split(x_train, y_train, groups=train_groups),
        scoring='accuracy',
        n_jobs=-1
    )
    
    return scores.mean()

study_rf = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=22)
)
study_rf.optimize(objective_rf, n_trials=100, show_progress_bar=True)

# Train final model with best params
best_rf = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=study_rf.best_params['n_estimators'],
        max_depth=study_rf.best_params['max_depth'],
        min_samples_leaf=study_rf.best_params['min_samples_leaf'],
        max_features=study_rf.best_params['max_features'],
        class_weight='balanced',
        random_state=22
    ))
])
best_rf.fit(x_train, y_train)

print(f"Best params: {study_rf.best_params}")
print(f"Best VALIDATION score: {study_rf.best_value:.3f}")
print(f"Training score: {best_rf.score(x_train, y_train):.3f}")
print(f"Overfit gap: {best_rf.score(x_train, y_train) - study_rf.best_value:.3f}")

# ============================================
# SUMMARY: COMPARE ALL MODELS
# ============================================
print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)

models_summary = {
    'Logistic Regression': (study_logreg.best_value, best_logreg.score(x_train, y_train)),
    'Linear SVC': (study_svc.best_value, best_svc.score(x_train, y_train)),
    'Naive Bayes': (study_nb.best_value, best_nb.score(x_train, y_train)),
    'Random Forest': (study_rf.best_value, best_rf.score(x_train, y_train))
}

for model_name, (val_score, train_score) in models_summary.items():
    print(f"\n{model_name}:")
    print(f"  Validation: {val_score:.3f}")
    print(f"  Training: {train_score:.3f}")
    print(f"  Overfit gap: {train_score - val_score:.3f}")

# Find best model
best_model_name = max(models_summary.items(), key=lambda x: x[1][0])[0]
print(f"\n{'='*60}")
print(f"BEST MODEL: {best_model_name}")
print(f"{'='*60}")

# ============================================
# OPTIONAL: VISUALIZE OPTIMIZATION HISTORY
# ============================================
print("\n### OPTIONAL: Visualization Code ###")
print("# Uncomment to visualize optimization history:")
print("""
import optuna.visualization as vis
import matplotlib.pyplot as plt

# Optimization history for Random Forest (best model)
fig = vis.plot_optimization_history(study_rf)
fig.show()

# Parameter importance for Random Forest
fig = vis.plot_param_importances(study_rf)
fig.show()

# Parallel coordinate plot for Random Forest
fig = vis.plot_parallel_coordinate(study_rf)
fig.show()
""")


c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-11-19 11:36:19,007] A new study created in memory with name: no-name-b19c5cde-c973-4cfc-bb70-a0e53e35d5a9


TESTING MODELS WITH OPTUNA OPTIMIZATION

### 1. LOGISTIC REGRESSION ###


Best trial: 0. Best value: 0.635717:   2%|▏         | 1/50 [00:08<06:42,  8.21s/it]

[I 2025-11-19 11:36:27,211] Trial 0 finished with value: 0.6357174988753936 and parameters: {'C': 0.006820907334120959}. Best is trial 0 with value: 0.6357174988753936.


Best trial: 1. Best value: 0.667072:   4%|▍         | 2/50 [00:11<04:20,  5.42s/it]

...

[I 2025-11-19 11:38:49,090] Trial 97 finished with value: 0.6842555105713 and parameters: {'n_estimators': 198, 'max_depth': 16, 'min_samples_leaf': 9, 'max_features': 'sqrt'}. Best is trial 96 with value: 0.6842555105713.


Best trial: 96. Best value: 0.684256:  99%|█████████▉| 99/100 [01:30<00:00,  1.09it/s]

[I 2025-11-19 11:38:49,988] Trial 98 finished with value: 0.6548358074673865 and parameters: {'n_estimators': 197, 'max_depth': 17, 'min_samples_leaf': 9, 'max_features': 'log2'}. Best is trial 96 with value: 0.6842555105713.


Best trial: 96. Best value: 0.684256: 100%|██████████| 100/100 [01:31<00:00,  1.10it/s]


[I 2025-11-19 11:38:50,890] Trial 99 finished with value: 0.6704903283850652 and parameters: {'n_estimators': 190, 'max_depth': 18, 'min_samples_leaf': 8, 'max_features': 'sqrt'}. Best is trial 96 with value: 0.6842555105713.
Best params: {'n_estimators': 196, 'max_depth': 16, 'min_samples_leaf': 9, 'max_features': 'sqrt'}
Best VALIDATION score: 0.684
Training score: 0.774
Overfit gap: 0.090

MODEL COMPARISON SUMMARY

Logistic Regression:
  Validation: 0.669
  Training: 0.700
  Overfit gap: 0.031

Linear SVC:
  Validation: 0.679
  Training: 0.752
  Overfit gap: 0.073

Naive Bayes:
  Validation: 0.663
  Training: 0.762
  Overfit gap: 0.099

Random Forest:
  Validation: 0.684
  Training: 0.774
  Overfit gap: 0.090

BEST MODEL: Random Forest

### OPTIONAL: Visualization Code ###
# Uncomment to visualize optimization history:

import optuna.visualization as vis
import matplotlib.pyplot as plt

# Optimization history for Random Forest (best model)
fig = vis.plot_optimization_history(study_r

Logistic Regression below

In [46]:
from sklearn.model_selection import GroupKFold


best_tfidf = grid_search.best_estimator_

grid_log_reg = {'classifier__C': [0.001, 0.01, 0.1, 1.]}
cv1 = GroupKFold(n_splits=5)
grid_search_logreg = GridSearchCV(best_tfidf, grid_log_reg,cv=cv1,scoring='accuracy', n_jobs=-1,verbose=2)
grid_search_logreg.fit(x_train, y_train, groups = train_group)


print(f"Best log reg model parameters: {grid_search_logreg.best_params_}")
print(f"Best log reg model: {grid_search_logreg.best_estimator_}")
print(f"Best log reg model score: {grid_search_logreg.best_score_}")
best_model = grid_search_logreg.best_estimator_
train_acc = best_model.score(x_train, y_train)
print(train_acc)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV] END ................................classifier__C=0.001; total time=   0.1s
[CV] END ................................classifier__C=0.001; total time=   0.1s
[CV] END ................................classifier__C=0.001; total time=   0.1s
[CV] END ................................classifier__C=0.001; total time=   0.1s
[CV] END ................................classifier__C=0.001; total time=   0.1s
[CV] END .................................classifier__C=0.01; total time=   0.1s
[CV] END .................................classifier__C=0.01; total time=   0.1s
[CV] END .................................classifier__C=0.01; total time=   0.1s
[CV] END .................................classifier__C=0.01; total time=   0.1s
[CV] END ..................................classifier__C=0.1; total time=   0.1s
[CV] END .................................classifier__C=0.01; total time=   0.2s
[CV] END ..................................classi

In [47]:
# Trying linear SVC
from sklearn.svm import LinearSVC

best_preprocessor = best_tfidf.named_steps['preprocessor']

full_pipeline2 = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', LinearSVC(dual=True))
])

grid_lin_svc= {'classifier__C': [0.01, 0.1, 1.0, 10.0], 'classifier__max_iter': [5000, 10000, 20000]}
grid_search_lin_svc = GridSearchCV(full_pipeline2, grid_lin_svc,cv=cv1,scoring='accuracy', n_jobs=-1,verbose=2)
grid_search_lin_svc.fit(x_train, y_train, groups =train_group)

print(f"Best log reg model parameters: {grid_search_lin_svc.best_params_}")
print(f"Best log reg model: {grid_search_lin_svc.best_estimator_}")
print(f"Best log reg model score: {grid_search_lin_svc.best_score_}")
best_model_svc = grid_search_lin_svc.best_estimator_
train_acc_svc = best_model_svc.score(x_train, y_train)
print(train_acc_svc)




Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END ......classifier__C=0.01, classifier__max_iter=5000; total time=   0.1s
[CV] END ......classifier__C=0.01, classifier__max_iter=5000; total time=   0.1s
[CV] END ......classifier__C=0.01, classifier__max_iter=5000; total time=   0.1s
[CV] END ......classifier__C=0.01, classifier__max_iter=5000; total time=   0.1s
[CV] END .....classifier__C=0.01, classifier__max_iter=10000; total time=   0.1s
[CV] END .....classifier__C=0.01, classifier__max_iter=10000; total time=   0.1s
[CV] END ......classifier__C=0.01, classifier__max_iter=5000; total time=   0.2s
[CV] END .....classifier__C=0.01, classifier__max_iter=10000; total time=   0.1s
[CV] END .....classifier__C=0.01, classifier__max_iter=10000; total time=   0.1s
[CV] END .....classifier__C=0.01, classifier__max_iter=20000; total time=   0.1s
[CV] END .....classifier__C=0.01, classifier__max_iter=20000; total time=   0.1s
[CV] END .....classifier__C=0.01, classifier__ma

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END ......classifier__C=10.0, classifier__max_iter=5000; total time=   0.6s
[CV] END ......classifier__C=10.0, classifier__max_iter=5000; total time=   0.6s
[CV] END ......classifier__C=10.0, classifier__max_iter=5000; total time=   0.6s
[CV] END ......classifier__C=10.0, classifier__max_iter=5000; total time=   0.6s
[CV] END .....classifier__C=10.0, classifier__max_iter=10000; total time=   0.7s
[CV] END .....classifier__C=10.0, classifier__max_iter=10000; total time=   0.8s
[CV] END .....classifier__C=10.0, classifier__max_iter=10000; total time=   0.8s
[CV] END .....classifier__C=10.0, classifier__max_iter=10000; total time=   0.8s
[CV] END .....classifier__C=10.0, classifier__max_iter=10000; total time=   0.5s
[CV] END ......classifier__C=10.0, classifier__max_iter=5000; total time=   0.6s


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END .....classifier__C=10.0, classifier__max_iter=20000; total time=   0.7s
[CV] END .....classifier__C=10.0, classifier__max_iter=20000; total time=   0.7s
[CV] END .....classifier__C=10.0, classifier__max_iter=20000; total time=   0.5s
[CV] END .....classifier__C=10.0, classifier__max_iter=20000; total time=   0.7s
[CV] END .....classifier__C=10.0, classifier__max_iter=20000; total time=   0.7s
Best log reg model parameters: {'classifier__C': 0.1, 'classifier__max_iter': 5000}
Best log reg model: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='',
                                                                                 strategy='constant')),
                                                                  ('combiner',
                                           

In [48]:
# Trying Naive Bayes

from sklearn.naive_bayes import MultinomialNB

full_pipeline3 = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', MultinomialNB())
])

grid_lin_nb=  {'classifier__alpha': [0.1, 0.5, 1.0, 2.0]}
grid_search_nb = GridSearchCV(full_pipeline3, grid_lin_nb,cv=cv1,scoring='accuracy', n_jobs=-1,verbose=2)
grid_search_nb.fit(x_train, y_train, groups=train_group)

print(f"Best log reg model parameters: {grid_search_nb.best_params_}")
print(f"Best log reg model: {grid_search_nb.best_estimator_}")
print(f"Best log reg model score: {grid_search_nb.best_score_}")
best_model_nb = grid_search_nb.best_estimator_
train_acc_nb = grid_search_nb.score(x_train, y_train)
print(train_acc_nb)


Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV] END ..............................classifier__alpha=0.1; total time=   0.1s
[CV] END ..............................classifier__alpha=0.1; total time=   0.1s
[CV] END ..............................classifier__alpha=0.1; total time=   0.1s
[CV] END ..............................classifier__alpha=0.1; total time=   0.1s
[CV] END ..............................classifier__alpha=0.5; total time=   0.1s
[CV] END ..............................classifier__alpha=0.1; total time=   0.1s
[CV] END ..............................classifier__alpha=0.5; total time=   0.1s
[CV] END ..............................classifier__alpha=0.5; total time=   0.1s
[CV] END ..............................classifier__alpha=0.5; total time=   0.1s
[CV] END ..............................classifier__alpha=1.0; total time=   0.1s
[CV] END ..............................classifier__alpha=0.5; total time=   0.1s
[CV] END ..............................classifier

In [49]:
# Trying random forests

from sklearn.ensemble import RandomForestClassifier
full_pipeline4 = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', RandomForestClassifier(random_state=22))
])

grid_rf = {
    'classifier__n_estimators': [100, 300],
    'classifier__max_depth': [None, 10, 20],
    'classifier__max_features': ['sqrt', 'log2']
}

grid_search_rf = GridSearchCV(full_pipeline4, grid_rf,cv=cv1,scoring='accuracy', n_jobs=-1,verbose=2)
grid_search_rf.fit(x_train, y_train, groups=train_group)

print(f"Best log reg model parameters: {grid_search_rf.best_params_}")
print(f"Best log reg model: {grid_search_rf.best_estimator_}")
print(f"Best log reg model score: {grid_search_rf.best_score_}")
best_model_rf = grid_search_rf.best_estimator_
train_acc_rf = grid_search_rf.score(x_train, y_train)
print(train_acc_rf)


Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__n_estimators=100; total time=   0.3s
[CV] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__n_estimators=100; total time=   0.3s
[CV] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__n_estimators=100; total time=   0.3s
[CV] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__n_estimators=100; total time=   0.3s
[CV] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__n_estimators=100; total time=   0.4s
[CV] END classifier__max_depth=None, classifier__max_features=log2, classifier__n_estimators=100; total time=   0.2s
[CV] END classifier__max_depth=None, classifier__max_features=log2, classifier__n_estimators=100; total time=   0.2s
[CV] END classifier__max_depth=None, classifier__max_features=log2, classifier__n_estimators=100; total time=   0.2s
[CV

In [10]:
# TF=IDF DEVELOPMENT IN ISOLATION
"""CORPUS:
├── Document 1 (Row 1): col1 + col2 + col3 → Label: ChatGPT
├── Document 2 (Row 2): col1 + col2 + col3 → Label: Claude

Term frequency (TF): the raw count of a term in a document
Inverse document frequency (IDF): how important a word is, idf(t,D)=log(N/df(t))
"""
# GridSearchCV for hyperparam tuning
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

# TODO: TAKE OUT WHEN YOU MOVE TO PIPELINE
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='constant', fill_value='')
df[text_cols] = imputer.fit_transform(df[text_cols])

# make corpus, create new column
df['corpus'] = df[text_cols].agg(' '.join, axis=1)
corpus = df['corpus']
labels = df['target']

# Create pipeline
pipeline = Pipeline([
    # TODO: also try removing stop words and adding max_df
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000))
])

# Define parameter grid
param_grid = {
    'tfidf__max_features': [500, 1000, 2000, 3000, None], # max number of features
    'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)], # length of context range
    'tfidf__min_df': [1, 2, 3, 4, 5] # minimum number times word has to appear to be included
}

# Search
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')
grid_search.fit(corpus, labels)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best accuracy: {grid_search.best_score_:.3f}")




Best parameters: {'tfidf__max_features': None, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 3)}
Best accuracy: 0.605


WON'T BE RUNNING THIS BELOW

In [ ]:
# LLM EMBEDDINGS

import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
import pandas as pd
# use pretrained weights

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

def embed_texts(text_col, batch_size=32):
    """
    Accepts a pandas Series, list, numpy array, or single string and returns
    a numpy array of mean-pooled embeddings (N, hidden_size).
    Replaces missing values with empty strings so the tokenizer always gets str inputs.
    """
    # normalize input to a list of strings
    texts = [ '' if pd.isna(x) else str(x) for x in text_col ]

    if len(texts) == 0:
        return np.zeros((0, model.config.hidden_size))

    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        # pass to model with no gradient bc we are not training, just getting embeddings
        with torch.no_grad():
            # done per response in batch  ============================================
            outputs = model(**enc) # outputs.last_hidden_state: (B, seq_len, 768)

            # Mean Pooling: average vectors of each true token vector in the text ====
            mask = enc["attention_mask"].unsqueeze(-1) # (B, seq_len, 1), add 1 dim for multiplication
            masked = outputs.last_hidden_state * mask # ignores padding tokens
            summed = masked.sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1e-9)
            mean_pooled = summed / counts  # (B, hidden)

            all_embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(all_embeddings)

# embed_best_tasks_free = embed_texts(df['best_tasks_free'])
# print(embed_best_tasks_free)
# # embed_subopt_tasks_free = embed_texts(df['subopt_tasks_free'])
# embed_verify_method_free = embed_texts(df['verify_method_free'])

def embed_texts(text_cols, df, batch_size=32):
    col_embedding = []
    for col in text_cols:
        embed_texts(df[col], batch_size=batch_size)
        col_embedding.append(col)
    return np.hstack(col_embedding)

print(embed_texts(text_cols, df))


c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 